# Dock drugability-filtered ACE candidates into 1O86

Candidates are taken only from the consensus model + descriptor + Glaxo-alert filter. The 1O86 docking box and receptor prepared in the redocking notebook are reused unchanged.

> **Validation status: WAIVED.** Redocking of lisinopril into 1O86 with this flexible Vina protocol did **not** reproduce the crystal pose (see notebook 4). With `ALLOW_FAILED_REDOCKING = True` this notebook still runs, but every output is stamped *not validated*. Vina scores here are supporting, non-validated evidence: they cannot confirm that a compound is an ACE inhibitor, and Vina's typical error (around 2 kcal/mol, as I recall; check the literature) is larger than most differences between candidates. Three reference ACE inhibitors (lisinopril, captopril, enalaprilat) are docked with the identical protocol as a yardstick.

In [ ]:
from pathlib import Path
import os, sys
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    ROOT = Path('/content/drive/MyDrive/ACE_Workflow')
except ImportError:
    ROOT = Path(os.environ.get('ACE_WORKFLOW_ROOT', Path.cwd())).resolve()
    if not (ROOT / 'ace_config.py').exists(): ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'ace_config.py').exists()), ROOT)
if not (ROOT / 'ace_config.py').exists(): raise FileNotFoundError('Open the Google Drive-synced ACE_Workflow folder in VS Code.')
os.environ['ACE_WORKFLOW_ROOT'] = str(ROOT)
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from ace_config import *

import hashlib, json, re, shutil, subprocess
import numpy as np, pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem

VINA = shutil.which('vina') or shutil.which('vina.exe')
MEEKO_LIGAND = shutil.which('mk_prepare_ligand.py')
if not VINA or not MEEKO_LIGAND: raise RuntimeError('AutoDock Vina and Meeko are required on PATH.')
TOP_N_FOR_DOCKING = 100  # practical default; change before running to expand the campaign
SEEDS, EXHAUSTIVENESS, NUM_MODES = [7, 42, 2026], 16, 10
DOCKING_CPU = CPU_WORKERS  # CPU-only Vina run; no GPU is requested
candidates_file = PATHS['DRUGABILITY'] / 'ace_druggable_candidates.csv'
box_file = PATHS['REDOCKING_ANALYSIS'] / '1o86_docking_box.json'
redock_file = PATHS['REDOCKING_ANALYSIS'] / '1o86_redocking_summary.csv'
receptor_pdbqt = PATHS['REDOCKING_RECEPTOR'] / '1o86_receptor.pdbqt'
for required in [candidates_file, box_file, redock_file, receptor_pdbqt]:
    if not required.exists(): raise FileNotFoundError(f'Missing required upstream output: {required}')
ALLOW_FAILED_REDOCKING = True    # explicit waiver of the redocking gate; every output is stamped with the validation status
_redock = pd.read_csv(redock_file)
N_REDOCK_PASS = int((_redock.heavy_atom_rmsd_A <= 2.0).sum())
MEDIAN_REDOCK_RMSD = float(_redock.heavy_atom_rmsd_A.median())
if N_REDOCK_PASS >= 2:
    VALIDATION_STATUS = 'VALIDATED: redocking reproduced the crystal pose in >= 2 seeds'
else:
    VALIDATION_STATUS = (f'NOT VALIDATED (waived): lisinopril redocking failed, {N_REDOCK_PASS}/{len(_redock)} seeds within 2 A, '
                         f'median RMSD {MEDIAN_REDOCK_RMSD:.1f} A')
    if not ALLOW_FAILED_REDOCKING:
        raise RuntimeError('Redocking did not reproduce the 1O86 crystal pose. Set ALLOW_FAILED_REDOCKING = True only if you accept non-validated docking scores.')
    print('=' * 100); print('WARNING - ' + VALIDATION_STATUS)
    print('Docking scores below are NOT validated predictors of pose or affinity and cannot confirm eligibility.'); print('=' * 100)
box = json.loads(box_file.read_text())

In [ ]:
raw_candidates = pd.read_csv(candidates_file)
# Dock each chemical once, keep provenance, then rank by priority score.
agg = {'source': lambda s: ' | '.join(sorted(set(map(str, s)))),
       'source_id': lambda s: ' | '.join(sorted(set(map(str, s)))),
       'priority_score': 'max', 'ace_active_probability': 'max', 'predicted_pchembl': 'max',
       'predicted_ic50_nM': 'min', 'glaxo_alert_count': 'min'}
for extra in ('max_train_tanimoto_cls', 'max_train_tanimoto_reg'):   # applicability-domain signal
    if extra in raw_candidates.columns: agg[extra] = 'max'
agg = {k: v for k, v in agg.items() if k in raw_candidates.columns}
# groupby sorts by SMILES, so rank AFTER aggregating; otherwise head(N) would take the alphabetically first compounds.
candidates = (raw_candidates.groupby('canonical_smiles', as_index=False).agg(agg)
              .sort_values('priority_score', ascending=False).head(TOP_N_FOR_DOCKING).reset_index(drop=True))
candidates['candidate_id'] = [f'ACE_{i+1:04d}_{hashlib.sha1(s.encode()).hexdigest()[:8]}' for i, s in enumerate(candidates.canonical_smiles)]
candidates['is_control'] = False
# Reference ACE inhibitors docked with the identical protocol: a yardstick for what a 'good' score looks like in this setup.
CONTROLS = {'CTRL_lisinopril': 'NCCCC[C@H](N[C@@H](CCc1ccccc1)C(=O)O)C(=O)N1CCC[C@H]1C(=O)O',
            'CTRL_captopril': 'C[C@@H](CS)C(=O)N1CCC[C@H]1C(=O)O',
            'CTRL_enalaprilat': 'C[C@H](N[C@@H](CCc1ccccc1)C(=O)O)C(=O)N1CCC[C@H]1C(=O)O'}
controls = pd.DataFrame({'candidate_id': list(CONTROLS), 'canonical_smiles': list(CONTROLS.values()),
                         'source': 'reference ACE inhibitor (control)', 'is_control': True})
candidates = pd.concat([controls, candidates], ignore_index=True)
candidates.to_csv(PATHS['DOCKING_REPORTS'] / 'docking_input_candidates.csv', index=False)
print(f'Docking {len(candidates) - len(CONTROLS):,} unique candidates (top by priority score, from {len(raw_candidates):,} drugability-passing rows) plus {len(CONTROLS)} reference inhibitors.')

In [ ]:
# Generate reproducible 3D conformers and Meeko PDBQT inputs.
prepared = []
for row in candidates.itertuples(index=False):
    mol = Chem.AddHs(Chem.MolFromSmiles(row.canonical_smiles))
    params = AllChem.ETKDGv3(); params.randomSeed = RANDOM_STATE
    embedded = AllChem.EmbedMolecule(mol, params)
    if embedded != 0:
        params.useRandomCoords = True; embedded = AllChem.EmbedMolecule(mol, params)  # retry for hard macrocycles/cages
    if embedded != 0:
        prepared.append({'candidate_id': row.candidate_id, 'status': 'FAILED_EMBED'}); continue
    AllChem.MMFFOptimizeMolecule(mol, maxIters=500)
    sdf = PATHS['DOCKING_LIGANDS'] / f'{row.candidate_id}.sdf'
    writer = Chem.SDWriter(str(sdf)); writer.write(mol); writer.close()
    pdbqt = PATHS['DOCKING_PDBQT'] / f'{row.candidate_id}.pdbqt'
    result = subprocess.run([MEEKO_LIGAND, '-i', str(sdf), '-o', str(pdbqt)], text=True, capture_output=True)
    prepared.append({'candidate_id': row.candidate_id, 'status': 'READY' if result.returncode == 0 else 'FAILED_MEEKO', 'sdf': str(sdf), 'ligand_pdbqt': str(pdbqt), 'error': result.stderr[-500:] if result.returncode else ''})
prepared = pd.DataFrame(prepared)
prepared.to_csv(PATHS['DOCKING_REPORTS'] / 'ligand_preparation.csv', index=False)
display(prepared.status.value_counts())

In [ ]:
# Checkpoint each candidate/seed so an interrupted docking job can resume safely.
checkpoint_file = PATHS['DOCKING_RUNS'] / 'docking_checkpoint.csv'
checkpoint = pd.read_csv(checkpoint_file) if checkpoint_file.exists() else pd.DataFrame(columns=['candidate_id', 'seed', 'status', 'affinity_kcal_mol', 'pose_pdbqt', 'log_file', 'error'])
records = checkpoint[checkpoint.status.eq('DONE')].to_dict('records')  # failed pairs are retried, not duplicated
ready = prepared.query("status == 'READY'").set_index('candidate_id')
import time
n_total = len(ready) * len(SEEDS); n_done0 = len(records); t_start = time.time()
print(f'{n_total} candidate/seed runs in total ({n_done0} already done). Progress and a runtime estimate are printed every 5 runs.')
for candidate_id, item in ready.iterrows():
    for seed in SEEDS:
        if ((checkpoint.candidate_id == candidate_id) & (checkpoint.seed == seed) & (checkpoint.status == 'DONE')).any(): continue
        pose = PATHS['DOCKING_RUNS'] / f'{candidate_id}_seed_{seed}.pdbqt'
        log = PATHS['DOCKING_RUNS'] / f'{candidate_id}_seed_{seed}.log'
        cmd = [VINA, '--receptor', str(receptor_pdbqt), '--ligand', item.ligand_pdbqt,
               '--center_x', str(box['center_x']), '--center_y', str(box['center_y']), '--center_z', str(box['center_z']),
               '--size_x', str(box['size_x']), '--size_y', str(box['size_y']), '--size_z', str(box['size_z']),
               '--cpu', str(DOCKING_CPU), '--exhaustiveness', str(EXHAUSTIVENESS), '--num_modes', str(NUM_MODES), '--seed', str(seed), '--out', str(pose)]  # Vina >= 1.2 has no --log option
        result = subprocess.run(cmd, text=True, capture_output=True)
        log.write_text(result.stdout + result.stderr)  # keep the Vina console output as the run log
        affinity = next((float(x.split()[3]) for x in pose.read_text(errors='replace').splitlines() if x.startswith('REMARK VINA RESULT')), np.nan) if pose.exists() else np.nan
        records.append({'candidate_id': candidate_id, 'seed': seed, 'status': 'DONE' if result.returncode == 0 else 'FAILED', 'affinity_kcal_mol': affinity, 'pose_pdbqt': str(pose), 'log_file': str(log), 'error': result.stderr[-500:] if result.returncode else ''})
        checkpoint = pd.DataFrame(records); checkpoint.to_csv(checkpoint_file, index=False)
        n_new = len(records) - n_done0
        if n_new % 5 == 0:
            per_run = (time.time() - t_start) / n_new; left = n_total - len(records)
            print(f'{len(records)}/{n_total} runs done; {per_run:.0f} s/run so far; about {left * per_run / 3600:.1f} h remaining (estimate; ligand size varies).')
docking = pd.DataFrame(records); display(docking.status.value_counts())

In [ ]:
# Choose the strongest pose only when all seeds give a geometrically consistent first pose.
def coords(path):
    result, model = [], False
    for line in Path(path).read_text(errors='replace').splitlines():
        if line.startswith('MODEL'): model = True; continue
        if model and line.startswith('ENDMDL'): break
        if line.startswith(('ATOM', 'HETATM')) and line[77:79].strip().upper() not in {'H', 'HD', 'HS'}:
            result.append([float(line[30:38]), float(line[38:46]), float(line[46:54])])
    return np.asarray(result)
def rmsd(a, b): return float(np.sqrt(np.mean(np.sum((a-b)**2, axis=1))))
rep_rows = []
for candidate_id, group in docking.query("status == 'DONE'").groupby('candidate_id'):
    poses = list(group.pose_pdbqt)
    arrays = [coords(p) for p in poses]
    pairwise = [rmsd(arrays[i], arrays[j]) for i in range(len(arrays)) for j in range(i+1, len(arrays)) if len(arrays[i]) == len(arrays[j])]
    best = group.sort_values('affinity_kcal_mol').iloc[0]
    n_seeds_done, mean_aff, sd_aff = len(group), float(group.affinity_kcal_mol.mean()), float(group.affinity_kcal_mol.std(ddof=1)) if len(group) > 1 else np.nan
    reproducible = bool(pairwise) and max(pairwise) <= 2.0
    destination = PATHS['DOCKING_POSES'] / f'{candidate_id}_representative.pdbqt'
    shutil.copy2(best.pose_pdbqt, destination)
    rep_rows.append({'candidate_id': candidate_id, 'best_affinity_kcal_mol': best.affinity_kcal_mol, 'mean_affinity_kcal_mol': mean_aff, 'sd_affinity_kcal_mol': sd_aff, 'n_seeds_done': n_seeds_done, 'max_seed_rmsd_A': max(pairwise) if pairwise else np.nan, 'pose_reproducible_2A': reproducible, 'representative_pose_pdbqt': str(destination)})
representatives = pd.DataFrame(rep_rows)
integrated = candidates.merge(representatives, on='candidate_id', how='left').sort_values(['pose_reproducible_2A', 'best_affinity_kcal_mol'], ascending=[False, True])
# 'Acceptable' dG is decided BEFORE looking at the candidates: not worse than the reference inhibitor lisinopril plus a tolerance.
DG_THRESHOLD_KCAL = None      # set an absolute value (e.g. -7.0) to override; None = derive from the lisinopril control
DG_TOLERANCE_KCAL = 1.0       # heuristic tolerance; Vina differences below ~1 kcal/mol are within its usual error
ref = integrated.loc[integrated.candidate_id == 'CTRL_lisinopril', 'best_affinity_kcal_mol']
ref_score = float(ref.iloc[0]) if len(ref) and ref.notna().all() else np.nan
if DG_THRESHOLD_KCAL is None:
    if ref_score != ref_score: raise RuntimeError('The lisinopril control did not dock; set DG_THRESHOLD_KCAL manually.')
    DG_THRESHOLD_KCAL = ref_score + DG_TOLERANCE_KCAL
integrated['dg_vs_lisinopril_kcal'] = integrated.best_affinity_kcal_mol - ref_score
integrated['dg_acceptable'] = integrated.best_affinity_kcal_mol <= DG_THRESHOLD_KCAL
integrated['redocking_validation'] = VALIDATION_STATUS
integrated.to_csv(PATHS['DOCKING_REPORTS'] / 'ace_1o86_integrated_docking.csv', index=False)
ctrl = integrated[integrated.is_control == True]
print('Reference inhibitors (same protocol):'); display(ctrl[['candidate_id', 'best_affinity_kcal_mol', 'mean_affinity_kcal_mol', 'sd_affinity_kcal_mol', 'pose_reproducible_2A']].round(2))
cand_only = integrated[integrated.is_control != True]
print(f'Acceptable dG threshold: <= {DG_THRESHOLD_KCAL:.2f} kcal/mol (lisinopril control {ref_score:.2f} + tolerance {DG_TOLERANCE_KCAL}). '
      f'{int(cand_only.dg_acceptable.sum())}/{len(cand_only)} candidates meet it.')
print(VALIDATION_STATUS)
(PATHS['DOCKING_REPORTS'] / 'docking_disclaimer.txt').write_text(
    'AutoDock Vina docking was run without a validated protocol: ' + VALIDATION_STATUS + '.\n'
    'Scores (kcal/mol) are not validated predictors of binding pose or affinity and cannot by themselves confirm that a compound is an ACE inhibitor. '
    'They are reported as supporting, non-validated prioritisation evidence, alongside reference ACE inhibitors docked with the identical protocol. '
    'Experimental ACE inhibition assays are required for confirmation.\n')
display(integrated.head(25))

In [ ]:
# Optional PLIP complexes; docking score and PLIP contacts are prioritization evidence, not binding confirmation.
plip = shutil.which('plip')
receptor_pdb = PATHS['REDOCKING_RECEPTOR'] / '1o86_chainA_with_Zn.pdb'
plip_rows = []
for row in integrated.dropna(subset=['representative_pose_pdbqt']).itertuples(index=False):
    complex_pdb = PATHS['DOCKING_COMPLEXES'] / f'{row.candidate_id}_complex.pdb'
    ligand_pdb = PATHS['DOCKING_COMPLEXES'] / f'{row.candidate_id}_ligand.pdb'
    write_ligand_pdb(row.representative_pose_pdbqt, ligand_pdb)
    # The receptor file ends with END/TER records; parsers stop at END, so strip them or the ligand would be ignored.
    rec_lines = [l for l in receptor_pdb.read_text().splitlines() if not l.startswith(('END', 'TER'))]
    lig_lines = [l for l in ligand_pdb.read_text().splitlines() if l.startswith('HETATM')]
    complex_pdb.write_text('\n'.join(rec_lines + ['TER'] + lig_lines + ['END']) + '\n')
    if plip:
        result = subprocess.run([plip, '-f', str(complex_pdb), '-o', str(PATHS['DOCKING_COMPLEXES'] / row.candidate_id), '-x'], text=True, capture_output=True)
        plip_rows.append({'candidate_id': row.candidate_id, 'plip_status': 'DONE' if result.returncode == 0 else 'FAILED'})
pd.DataFrame(plip_rows).to_csv(PATHS['DOCKING_REPORTS'] / 'plip_status.csv', index=False)
(PATHS['DOCKING_REPORTS'] / 'docking_metadata.json').write_text(json.dumps({'receptor': str(receptor_pdbqt), 'receptor_sha256': sha256_file(receptor_pdbqt), 'box': box, 'seeds': SEEDS, 'exhaustiveness': EXHAUSTIVENESS, 'num_modes': NUM_MODES, 'top_n_for_docking': TOP_N_FOR_DOCKING, 'validation_status': VALIDATION_STATUS, 'redocking_seeds_within_2A': N_REDOCK_PASS, 'redocking_median_rmsd_A': MEDIAN_REDOCK_RMSD, 'dg_threshold_kcal': DG_THRESHOLD_KCAL, 'dg_tolerance_kcal': DG_TOLERANCE_KCAL, 'lisinopril_control_score_kcal': ref_score}, indent=2))